# Movie Recommendation System

**Project** : Movie Recommendation System  
**Team** : Syed Asjad Ali Zaidi (22K-4234) | Affan Jan (22K-4475) | Muhammad Saad (22K-4407)  
**Dataset** : MovieLens 100K (ml-100k) — GroupLens Research  
**Technique**: Item-Item Collaborative Filtering + Cosine Similarity

---

This notebook implements a movie recommendation engine using **Item-Item Collaborative Filtering**. The system leverages the "wisdom of the crowd" — it finds movies that are similar to a given movie based on how users have rated them, using **Cosine Similarity** as the distance metric.

## Section 1: Imports and Configuration

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split

%matplotlib inline

# ── Configuration ─────────────────────────────────────
MIN_RATINGS_THRESHOLD = 50   # Minimum number of ratings a movie must have
TOP_N = 10                   # Default number of recommendations
RANDOM_STATE = 42            # For reproducible train/test splits

print('All imports loaded successfully.')

## Section 2: Data Loading and Preprocessing

In [ ]:
# Load ratings data
r_cols = ['user_id', 'movie_id', 'rating', 'unix_timestamp']
ratings = pd.read_csv('u.data', sep='\t', names=r_cols, encoding='latin-1')

# Load movie metadata
m_cols = ['movie_id', 'title', 'release_date', 'video_release_date', 'imdb_url']
movies = pd.read_csv('u.item', sep='|', names=m_cols, usecols=range(5),
                     encoding='latin-1')

print(f'Ratings loaded: {ratings.shape[0]} rows')
print(f'Movies loaded : {movies.shape[0]} rows')
ratings.head()

In [ ]:
movies.head()

In [ ]:
# Drop columns that are not needed
ratings.drop('unix_timestamp', axis=1, inplace=True)
movies.drop(['release_date', 'video_release_date', 'imdb_url'], axis=1, inplace=True)

# Merge movies and ratings
merged_df = pd.merge(movies, ratings, on='movie_id')

# Handle missing values
merged_df.dropna(subset=['title', 'rating'], inplace=True)

print(f'Merged dataset: {merged_df.shape[0]} rows, {merged_df.shape[1]} columns')
print(f'Columns: {list(merged_df.columns)}')
merged_df.info()

In [ ]:
# Compute per-movie rating statistics
movie_stats = merged_df.groupby('title').agg(
    num_ratings=('rating', 'count'),
    mean_rating=('rating', 'mean')
).reset_index()

print(f'Total unique movies: {movie_stats.shape[0]}')
print(f'Movies with >= {MIN_RATINGS_THRESHOLD} ratings: '
      f'{(movie_stats["num_ratings"] >= MIN_RATINGS_THRESHOLD).sum()}')

movie_stats.sort_values('mean_rating', ascending=False).head(10)

In [ ]:
# Apply minimum-ratings threshold filter
popular_movies = movie_stats[movie_stats['num_ratings'] >= MIN_RATINGS_THRESHOLD]['title']
filtered_df = merged_df[merged_df['title'].isin(popular_movies)].copy()

print(f'Filtered dataset: {filtered_df.shape[0]} ratings across '
      f'{filtered_df["title"].nunique()} movies')

In [ ]:
# Visualize the rating distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(filtered_df['rating'], bins=5, edgecolor='black', alpha=0.7)
axes[0].set_title("Distribution of Ratings")
axes[0].set_xlabel('Rating (1-5)')
axes[0].set_ylabel('Count')

rating_counts_per_movie = filtered_df.groupby('title')['rating'].count()
axes[1].hist(rating_counts_per_movie, bins=30, edgecolor='black', alpha=0.7)
axes[1].set_title('Ratings per Movie (After Filtering)')
axes[1].set_xlabel('Number of Ratings')
axes[1].set_ylabel('Number of Movies')

plt.tight_layout()
plt.show()

## Section 3: Pivot Table Construction

In [ ]:
# Build the Movies x Users pivot table
# Rows = movie titles, Columns = user IDs, Values = ratings
ratings_matrix = filtered_df.pivot_table(
    index='title', columns='user_id', values='rating'
)
ratings_matrix.fillna(0, inplace=True)

print(f'Pivot table shape: {ratings_matrix.shape}')
print(f'  → {ratings_matrix.shape[0]} movies × {ratings_matrix.shape[1]} users')
ratings_matrix.iloc[:5, :10]

## Section 4: Similarity Computation

In [ ]:
# Compute pairwise Cosine Similarity across all movie vectors
similarity_matrix = cosine_similarity(ratings_matrix.values)

# Store as a labeled DataFrame
similarity_df = pd.DataFrame(
    similarity_matrix,
    index=ratings_matrix.index,
    columns=ratings_matrix.index
)

# Zero out diagonal (a movie is always 1.0 similar to itself)
np.fill_diagonal(similarity_df.values, 0)

print(f'Similarity matrix shape: {similarity_df.shape}')
similarity_df.iloc[:5, :5]

## Section 5: Recommendation Function

In [ ]:
def get_recommendations(movie_title, sim_df=similarity_df, top_n=TOP_N):
    """
    Given a movie title, return the top-N most similar movies
    ranked by cosine similarity score (descending),
    excluding the input movie itself.

    Parameters
    ----------
    movie_title : str
        Exact title as it appears in the dataset.
    sim_df : pd.DataFrame
        Precomputed similarity matrix.
    top_n : int
        Number of recommendations.

    Returns
    -------
    pd.DataFrame with columns [rank, title, similarity_score]
    """
    if movie_title not in sim_df.index:
        print(f"'{movie_title}' not found in the filtered database.")
        return pd.DataFrame(columns=['rank', 'title', 'similarity_score'])

    sim_scores = sim_df[movie_title].sort_values(ascending=False).head(top_n)

    return pd.DataFrame({
        'rank': range(1, len(sim_scores) + 1),
        'title': sim_scores.index,
        'similarity_score': sim_scores.values
    })

## Section 6: Evaluation — Precision@K (K=10)

Since this is an **item-ranking** system (not a rating-prediction system), we evaluate using **Precision@K**.  

**Method**:  
1. Split ratings 80/20 into train and test sets.  
2. Build the similarity model on the **training** set only.  
3. For each user in the test set, identify their *relevant* movies (those rated ≥ 4).  
4. For each relevant movie, generate top-K recommendations from the train-based model.  
5. Precision@K = (# of recommended movies that are also relevant) / K.

In [ ]:
# ── Train / Test Split ────────────────────────────────
train_data, test_data = train_test_split(
    filtered_df, test_size=0.2, random_state=RANDOM_STATE
)

print(f'Training set : {train_data.shape[0]} ratings')
print(f'Test set     : {test_data.shape[0]} ratings')

In [ ]:
# ── Build model on training data ─────────────────────
train_matrix = train_data.pivot_table(
    index='title', columns='user_id', values='rating'
)
train_matrix.fillna(0, inplace=True)

train_sim = cosine_similarity(train_matrix.values)
train_sim_df = pd.DataFrame(
    train_sim,
    index=train_matrix.index,
    columns=train_matrix.index
)
np.fill_diagonal(train_sim_df.values, 0)

print(f'Train similarity matrix: {train_sim_df.shape}')

In [ ]:
# ── Compute Precision@K ──────────────────────────────
K = 10
RELEVANCE_THRESHOLD = 4   # A movie is "relevant" if rated >= 4

precisions = []

# Get users who appear in both train and test
test_users = test_data['user_id'].unique()

for user_id in test_users:
    # Movies this user rated highly in the TEST set (ground truth)
    user_test = test_data[test_data['user_id'] == user_id]
    relevant_movies = set(
        user_test[user_test['rating'] >= RELEVANCE_THRESHOLD]['title']
    )

    if len(relevant_movies) == 0:
        continue  # skip users with no relevant test items

    # Movies this user rated highly in the TRAIN set
    user_train = train_data[train_data['user_id'] == user_id]
    liked_train = user_train[user_train['rating'] >= RELEVANCE_THRESHOLD]['title'].tolist()

    if len(liked_train) == 0:
        continue

    # Generate recommendations based on the user's liked training movies
    rec_scores = pd.Series(0.0, index=train_sim_df.index)
    for movie in liked_train:
        if movie in train_sim_df.index:
            rec_scores += train_sim_df[movie]

    # Remove movies already seen in training
    seen_movies = set(user_train['title'])
    rec_scores = rec_scores.drop(
        labels=[m for m in seen_movies if m in rec_scores.index],
        errors='ignore'
    )

    # Top-K recommendations
    top_k_recs = set(rec_scores.sort_values(ascending=False).head(K).index)

    # Precision@K for this user
    hits = len(top_k_recs & relevant_movies)
    precisions.append(hits / K)

mean_precision_at_k = np.mean(precisions) if precisions else 0.0

print(f'Evaluation Results')
print(f'─' * 40)
print(f'Metric          : Precision@{K}')
print(f'Users evaluated : {len(precisions)}')
print(f'Mean Precision@{K}: {mean_precision_at_k:.4f}')

## Section 7: Demo / Results

Demonstrating the recommendation function with three different movies.

In [ ]:
demo_movies = [
    'Toy Story (1995)',
    'Star Wars (1977)',
    'Fargo (1996)'
]

for movie in demo_movies:
    print(f'\n{"═" * 60}')
    print(f'  Recommendations for: {movie}')
    print(f'{"═" * 60}')
    result = get_recommendations(movie)
    if not result.empty:
        print(result.to_string(index=False))
    print()